In [ ]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from typing import Annotated
from typing_extensions import TypedDict
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

# ===== Stateクラスの定義 =====
class State(TypedDict):
    messages: Annotated[list, add_messages]

# ===== グラフの構築 =====
def build_graph(model_name):
    # グラフのインスタンスを作成
    graph_builder = StateGraph(State)

    # 言語モデルの定義とツール定義の紐付け
    llm = ChatOpenAI(model_name=model_name)
    tool = TavilySearchResults(max_results=2)
    tools = [tool]
    llm_with_tools = llm.bind_tools(tools)

    # チャットボットノードの作成と追加
    def chatbot(state: State):
        return {"messages": [llm_with_tools.invoke(state["messages"])]}

    graph_builder.add_node("chatbot", chatbot)

    # ツールノードの作成と追加
    tool_node = ToolNode(tools)
    graph_builder.add_node("tools", tool_node)

    # 条件付エッジの作成と追加
    graph_builder.add_conditional_edges(
        "chatbot",
        tools_condition,
    )
    graph_builder.add_edge("tools", "chatbot")

    # 開始ノードの指定
    graph_builder.set_entry_point("chatbot")

    # 記憶を持つ実行可能なステートグラフの作成
    memory = MemorySaver()
    graph = graph_builder.compile(checkpointer=memory)

    return graph

# ===== グラフ実行関数 =====
def stream_graph_updates(graph: StateGraph, user_input: str):
    events = graph.stream(
        {"messages": [("user", user_input)]},
        {"configurable": {"thread_id": "1"}},
        stream_mode="values"
    )
    # 結果をストリーミングで得る
    for event in events:
        print(event["messages"][-1].content, flush=True)

# ===== メイン実行ロジック =====
# 環境変数の読み込み
load_dotenv("../.env")
os.environ['OPENAI_API_KEY'] = os.environ['API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini"

# グラフの作成
graph = build_graph(MODEL_NAME)

# メインループ
while True:
    user_input = input("質問:")
    if user_input.strip()=="":
        print("ありがとうございました!")
        break
    stream_graph_updates(graph, user_input)

こんにちは！
こんにちは！今日はどんなことをお手伝いできますか？
1たす2は？
1たす2は3です。何か他にお手伝いできることはありますか？
台湾観光について検索結果を教えて

[{"url": "https://www.knt.co.jp/travelguide/kaigai/027/", "content": "※新型コロナウイルス感染拡大に伴い、一部施設の利用中止や営業時間の変更、ご提供内容の変更が発生する場合があります。  \n※記載の情報は随時更新を行なっておりますが、予告なく変更となる場合がございます。あくまでも観光の参考としてご覧ください。  \n※ご利用には別途代金のかかるサービスもございます。現地にてご確認ください。\n\n※2022年11月1日時点の情報です。  \n※最新情報は各施設の公式サイトをご確認ください。\n\n## 関連特集\n\n台湾特集\n\n台北で人気のホテル14選をご紹介！\n\n台湾のおすすめグルメ7選！\n\n台湾のおすすめスイーツ6選！\n\nエバー航空で行く台湾\n\nアジア女子旅 おすすめの国\n\n合わせて読みたい\n\n ハワイ・オアフ島を観光するなら押さえておきたいスポット17選\n ハワイの海を満喫できるおすすめビーチ13選！穴場スポットも紹介\n ハワイで必ず行きたいグルメスポット！選りすぐりの8店を一挙紹介！\n\n## ほかの観光地を探す\n\n ハワイ\n グアム\n 韓国\n 台湾\n 香港\n オーストラリア\n シンガポール\n タイ\n ベトナム\n 北米・カナダ\n\n近畿日本ツーリスト 観光ガイドTOP\n\n## 国内ツアーランキング\n\n### エリアからツアーを探す\n\n ハワイ\n ホノルル\n グアム\n サイパン\n シドニー\n ケアンズ\n ロサンゼルス\n ニューヨーク\n ソウル\n 釜山\n 台湾\n シンガポール\n バンコク\n ホーチミン\n ダナン\n ハノイ\n 香港\n 香港ディズニーランド・リゾート\n\n### 目的・テーマからツアーを探す\n\n#### 季節・話題の海外旅行\n\n 春休み\n ゴールデンウィーク（GW）\n 夏休み・お盆\n 秋の連休\n 年末年始・お正月\n\n#### 連休を利用した海外旅行\n\n 1月\n 2月\n 3月